## Student Attendance Analytics with Pandas

In [1]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [2]:
df = pd.read_excel("aids_attendance(1).xlsx")
df

,Roll No,Student Name,Subject,Total Classes,Present,Absent,Excused,Attendance %,Status,Classes Needed (75%),Last Present Date,Consecutive Absences
0,4MW23AD50,SRAJAN S,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11,0,0%,Critical,33 more,Never,11
1,4MW24AD001,ADITHYA S POOJARY,Data Science & GenAI - 3nd Yr- AI&DS,11,7,4,0,64%,Critical,5 more,2026-10-07,0
2,4MW24AD002,ANKITHA PRASANNA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,100%,Safe,✓ Met,2026-10-07,0
3,4MW24AD003,ANUSHA KOTEGAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91%,Safe,✓ Met,2026-10-07,0
4,4MW24AD004,ANVITA RAMESH NAIK,Data Science & GenAI - 3nd Yr- AI&DS,11,9,2,0,82%,Warning,✓ Met,2026-10-07,0
...,...,...,...,...,...,...,...,...,...,...,...,...
57,4MW24AD059,ZEHA AYESHA,Data Science & GenAI - 3nd Yr- AI&DS,10,9,1,0,90%,Safe,✓ Met,2026-10-07,0
58,4MW24AD060,AMITH G PALANDYE,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0%,Critical,30 more,Never,10
59,4MW24AD061,AMOGH BHASKAR NAIK,Data Science & GenAI - 3nd Yr- AI&DS,10,5,5,0,50%,Critical,10 more,2026-10-07,0
60,4MW24AD062,CHANDISH,Data Science & GenAI - 3nd Yr- AI&DS,10,6,4,0,60%,Critical,6 more,2026-10-05,1


#### Section 1: Data Inspection

In [3]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""
print("First 10 rows:")
display(df.head(10))

print("DataFrame shape (rows, columns):", df.shape)
print("\nData types:")
print(df.dtypes)

First 10 rows:


,Roll No,Student Name,Subject,Total Classes,Present,Absent,Excused,Attendance %,Status,Classes Needed (75%),Last Present Date,Consecutive Absences
0,4MW23AD50,SRAJAN S,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11,0,0%,Critical,33 more,Never,11
1,4MW24AD001,ADITHYA S POOJARY,Data Science & GenAI - 3nd Yr- AI&DS,11,7,4,0,64%,Critical,5 more,2026-10-07,0
2,4MW24AD002,ANKITHA PRASANNA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,100%,Safe,✓ Met,2026-10-07,0
3,4MW24AD003,ANUSHA KOTEGAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91%,Safe,✓ Met,2026-10-07,0
4,4MW24AD004,ANVITA RAMESH NAIK,Data Science & GenAI - 3nd Yr- AI&DS,11,9,2,0,82%,Warning,✓ Met,2026-10-07,0
5,4MW24AD005,ARPITHA M THANTRY,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3,0,73%,Critical,1 more,2026-10-07,0
6,4MW24AD007,ASNIYA,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91%,Safe,✓ Met,2026-10-07,0
7,4MW24AD008,B. G PRARTHANA HEBBAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91%,Safe,✓ Met,2026-10-07,0
8,4MW24AD009,BHARATHRAJ A NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,4,7,0,36%,Critical,17 more,2026-10-05,1
9,4MW24AD010,BHEEMA KATTIMANI,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3,0,73%,Critical,1 more,2026-10-07,0


DataFrame shape (rows, columns): (62, 12)

Data types:
Roll No                 object
Student Name            object
Subject                 object
Total Classes            int64
Present                  int64
Absent                   int64
Excused                  int64
Attendance %            object
Status                  object
Classes Needed (75%)    object
Last Present Date       object
Consecutive Absences     int64
dtype: object


In [4]:
"""Q2. Check for missing values in each column. Identify columns with null values and count "Never" in Last Present Date."""
print("Missing (null) values in each column:")
print(df.isnull().sum())

never_count = df["Last Present Date"].astype(str).str.strip().str.casefold().eq("never").sum()
print("\n'Never' values in Last Present Date:", never_count)

columns_with_nulls = df.columns[df.isnull().any()].tolist()
print("Columns containing actual null values:", columns_with_nulls)

Missing (null) values in each column:
Roll No                 0
Student Name            0
Subject                 0
Total Classes           0
Present                 0
Absent                  0
Excused                 0
Attendance %            0
Status                  0
Classes Needed (75%)    0
Last Present Date       0
Consecutive Absences    0
dtype: int64

'Never' values in Last Present Date: 5
Columns containing actual null values: []


In [5]:
"""Q3. Rename the columns to snake_case and display the updated column list."""
df.rename(columns={
    "Roll No": "roll_no",
    "Student Name": "student_name",
    "Subject": "subject",
    "Total Classes": "total_classes",
    "Present": "present",
    "Absent": "absent",
    "Excused": "excused",
    "Attendance %": "attendance_pct",
    "Status": "status",
    "Classes Needed (75%)": "classes_needed_75",
    "Last Present Date": "last_present_date",
    "Consecutive Absences": "consecutive_absences"
}, inplace=True)

print("Updated columns:")
print(df.columns.tolist())

Updated columns:
['roll_no', 'student_name', 'subject', 'total_classes', 'present', 'absent', 'excused', 'attendance_pct', 'status', 'classes_needed_75', 'last_present_date', 'consecutive_absences']


In [6]:
"""Q4. Convert attendance percentage to numeric and Last Present Date to datetime; treat "Never" as NaT."""
df["attendance_pct"] = (
    df["attendance_pct"].astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
    .replace("", np.nan)
    .astype(float)
)

df["last_present_date"] = pd.to_datetime(
    df["last_present_date"].replace(r"(?i)^never$", pd.NA, regex=True),
    errors="coerce"
)

print(df[["attendance_pct", "last_present_date"]].head())
print("\nAttendance percentage data type:", df["attendance_pct"].dtype)
print("Last present date data type:", df["last_present_date"].dtype)

   attendance_pct last_present_date
0             0.0               NaT
1            64.0        2026-10-07
2           100.0        2026-10-07
3            91.0        2026-10-07
4            82.0        2026-10-07

Attendance percentage data type: float64
Last present date data type: datetime64[ns]


#### Section 2: Filtering & Selection

In [7]:
"""Q5. Filter Critical students and sort by attendance percentage ascending."""
critical_students = df.loc[df["status"].eq("Critical")].sort_values(
    by="attendance_pct", ascending=True
)
display(critical_students)

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75,last_present_date,consecutive_absences
0,4MW23AD50,SRAJAN S,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11,0,0.0,Critical,33 more,NaT,11
52,4MW24AD054,UJWAL. K,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0.0,Critical,30 more,NaT,10
58,4MW24AD060,AMITH G PALANDYE,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0.0,Critical,30 more,NaT,10
36,4MW24AD037,RISHAN FRANCIS MENEZES,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0.0,Critical,30 more,NaT,10
45,4MW24AD046,SHARANU PATIL,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0.0,Critical,30 more,NaT,10
61,4MW25AD400,SAMPATH S HEBBAR,Data Science & GenAI - 3nd Yr- AI&DS,10,1,9,0,10.0,Critical,26 more,2026-09-02,8
26,4MW24AD027,PRAJWAL,Data Science & GenAI - 3nd Yr- AI&DS,11,3,8,0,27.0,Critical,21 more,2026-10-05,1
30,4MW24AD031,PRAVEEN KUMAR N,Data Science & GenAI - 3nd Yr- AI&DS,10,3,7,0,30.0,Critical,18 more,2026-10-07,0
8,4MW24AD009,BHARATHRAJ A NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,4,7,0,36.0,Critical,17 more,2026-10-05,1
31,4MW24AD032,PREETH SHETTY,Data Science & GenAI - 3nd Yr- AI&DS,10,5,5,0,50.0,Critical,10 more,2026-10-05,1


In [8]:
"""Q6. Display students with zero present days."""
fully_absent = df.loc[df["present"].eq(0),
                      ["roll_no", "student_name", "consecutive_absences"]]
display(fully_absent)

,roll_no,student_name,consecutive_absences
0,4MW23AD50,SRAJAN S,11
36,4MW24AD037,RISHAN FRANCIS MENEZES,10
45,4MW24AD046,SHARANU PATIL,10
52,4MW24AD054,UJWAL. K,10
58,4MW24AD060,AMITH G PALANDYE,10


In [9]:
"""Q7(a). Display Safe students using .loc.
Q7(b). Display the first 15 rows and columns 2-6 using .iloc (positions 2 through 6 inclusive)."""
safe_students = df.loc[df["status"].eq("Safe")]
display(safe_students)

display(df.iloc[:15, 2:7])

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75,last_present_date,consecutive_absences
2,4MW24AD002,ANKITHA PRASANNA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,100.0,Safe,✓ Met,2026-10-07,0
3,4MW24AD003,ANUSHA KOTEGAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91.0,Safe,✓ Met,2026-10-07,0
6,4MW24AD007,ASNIYA,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91.0,Safe,✓ Met,2026-10-07,0
7,4MW24AD008,B. G PRARTHANA HEBBAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91.0,Safe,✓ Met,2026-10-07,0
14,4MW24AD015,K MANYA RAO,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91.0,Safe,✓ Met,2026-10-07,0
15,4MW24AD016,KAVYA,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,100.0,Safe,✓ Met,2026-10-07,0
28,4MW24AD029,PRATHEEKSHA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,100.0,Safe,✓ Met,2026-10-07,0
32,4MW24AD033,PRIYADARSHINI,Data Science & GenAI - 3nd Yr- AI&DS,10,10,0,0,100.0,Safe,✓ Met,2026-10-07,0
33,4MW24AD034,RAKSHITHA (ASHOK KUMAR),Data Science & GenAI - 3nd Yr- AI&DS,10,10,0,0,100.0,Safe,✓ Met,2026-10-07,0
35,4MW24AD036,RAMITHA R RAO,Data Science & GenAI - 3nd Yr- AI&DS,10,9,1,0,90.0,Safe,✓ Met,2026-10-07,0


,subject,total_classes,present,absent,excused
0,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11,0
1,Data Science & GenAI - 3nd Yr- AI&DS,11,7,4,0
2,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0
3,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0
4,Data Science & GenAI - 3nd Yr- AI&DS,11,9,2,0
5,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3,0
6,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0
7,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0
8,Data Science & GenAI - 3nd Yr- AI&DS,11,4,7,0
9,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3,0


In [10]:
"""Q8. Find students with at least 5 consecutive absences."""
display(df.loc[df["consecutive_absences"].ge(5),
               ["student_name", "status", "last_present_date"]])

,student_name,status,last_present_date
0,SRAJAN S,Critical,NaT
36,RISHAN FRANCIS MENEZES,Critical,NaT
45,SHARANU PATIL,Critical,NaT
52,UJWAL. K,Critical,NaT
58,AMITH G PALANDYE,Critical,NaT
61,SAMPATH S HEBBAR,Critical,2026-09-02


### Section 3: Grouping & Aggregation 

In [11]:
"""Q9. Group by status: student count, average attendance, and maximum consecutive absences."""
status_summary = df.groupby("status").agg(
    student_count=("roll_no", "count"),
    average_attendance_pct=("attendance_pct", "mean"),
    max_consecutive_absences=("consecutive_absences", "max")
)
display(status_summary)

,student_count,average_attendance_pct,max_consecutive_absences
status,,,
Critical,32,49.875000,11
Safe,23,94.086957,1
Warning,7,81.428571,1


In [12]:
"""Q10. Create Attendance Category and count students in each category."""
df["attendance_category"] = np.select(
    [
        df["attendance_pct"].ge(90),
        df["attendance_pct"].ge(75),
        df["attendance_pct"].ge(60)
    ],
    ["Excellent", "Good", "Average"],
    default="Poor"
)

print("Student count by attendance category:")
display(df["attendance_category"].value_counts().reindex(
    ["Excellent", "Good", "Average", "Poor"], fill_value=0
))

Student count by attendance category:


attendance_category
Excellent    23
Good          7
Average      19
Poor         13
Name: count, dtype: int64

In [13]:
"""Q11. Use .agg() to calculate total classes, total present, and total absent."""
totals = df.agg({
    "total_classes": "sum",
    "present": "sum",
    "absent": "sum"
})
display(totals)

total_classes    650
present          454
absent           196
dtype: int64

In [14]:
"""Q12. Group by status: minimum, maximum, and mean Present; sum of Absent."""
status_attendance = df.groupby("status").agg(
    present_min=("present", "min"),
    present_max=("present", "max"),
    present_mean=("present", "mean"),
    absent_sum=("absent", "sum")
)
display(status_attendance)

,present_min,present_max,present_mean,absent_sum
status,,,,
Critical,0,8,5.312500,168
Safe,9,11,9.695652,14
Warning,8,9,8.714286,14


### Section 4: Feature Engineering & Transformation

In [15]:
"""Q13. Calculate Risk Score and display the 10 highest-risk students."""
df["risk_score"] = (
    df["absent"] * 0.4
    + df["consecutive_absences"] * 0.6
)

display(df.nlargest(10, "risk_score")[
    ["roll_no", "student_name", "absent", "consecutive_absences", "risk_score"]
])

,roll_no,student_name,absent,consecutive_absences,risk_score
0,4MW23AD50,SRAJAN S,11,11,11.0
36,4MW24AD037,RISHAN FRANCIS MENEZES,10,10,10.0
45,4MW24AD046,SHARANU PATIL,10,10,10.0
52,4MW24AD054,UJWAL. K,10,10,10.0
58,4MW24AD060,AMITH G PALANDYE,10,10,10.0
61,4MW25AD400,SAMPATH S HEBBAR,9,8,8.4
26,4MW24AD027,PRAJWAL,8,1,3.8
8,4MW24AD009,BHARATHRAJ A NAYAK,7,1,3.4
30,4MW24AD031,PRAVEEN KUMAR N,7,0,2.8
31,4MW24AD032,PREETH SHETTY,5,1,2.6


In [16]:
"""Q14. Extract numeric value from Classes Needed (75%); values such as "✓ Met" become 0."""
df["classes_needed_numeric"] = (
    df["classes_needed_75"].astype(str)
    .str.extract(r"(\d+)", expand=False)
    .fillna("0")
    .astype(int)
)

display(df[["classes_needed_75", "classes_needed_numeric"]])

,classes_needed_75,classes_needed_numeric
0,33 more,33
1,5 more,5
2,✓ Met,0
3,✓ Met,0
4,✓ Met,0
...,...,...
57,✓ Met,0
58,30 more,30
59,10 more,10
60,6 more,6


In [17]:
"""Q15. Create Attendance Flag using apply() and a lambda function."""
df["attendance_flag"] = df["attendance_pct"].apply(
    lambda attendance: "Defaulter" if attendance < 75 else "Safe"
)

display(df[["student_name", "attendance_pct", "attendance_flag"]])

,student_name,attendance_pct,attendance_flag
0,SRAJAN S,0.0,Defaulter
1,ADITHYA S POOJARY,64.0,Defaulter
2,ANKITHA PRASANNA NAYAK,100.0,Safe
3,ANUSHA KOTEGAR,91.0,Safe
4,ANVITA RAMESH NAIK,82.0,Safe
...,...,...,...
57,ZEHA AYESHA,90.0,Safe
58,AMITH G PALANDYE,0.0,Defaulter
59,AMOGH BHASKAR NAIK,50.0,Defaulter
60,CHANDISH,60.0,Defaulter


In [18]:
"""Q16. Bin attendance percentages into the requested ranges using pd.cut()."""
df["attendance_range"] = pd.cut(
    df["attendance_pct"],
    bins=[-0.001, 50, 75, 90, 100],
    labels=["Critical", "Warning", "Good", "Excellent"],
    include_lowest=True
)

display(df[["student_name", "attendance_pct", "attendance_range"]])

,student_name,attendance_pct,attendance_range
0,SRAJAN S,0.0,Critical
1,ADITHYA S POOJARY,64.0,Warning
2,ANKITHA PRASANNA NAYAK,100.0,Excellent
3,ANUSHA KOTEGAR,91.0,Excellent
4,ANVITA RAMESH NAIK,82.0,Good
...,...,...,...
57,ZEHA AYESHA,90.0,Good
58,AMITH G PALANDYE,0.0,Critical
59,AMOGH BHASKAR NAIK,50.0,Critical
60,CHANDISH,60.0,Warning


### Section 5: Advanced Analysis 

In [19]:
"""Q17. Pivot table counting students by Status and Attendance Category."""
attendance_pivot = pd.pivot_table(
    df,
    index="status",
    columns="attendance_category",
    values="roll_no",
    aggfunc="count",
    fill_value=0,
    observed=False
)
display(attendance_pivot)

attendance_category,Average,Excellent,Good,Poor
status,,,,
Critical,19,0,0,13
Safe,0,23,0,0
Warning,0,0,7,0


In [20]:
"""Q18. Extract base name (first name) and display all rows where the base name repeats."""
df["base_name"] = (
    df["student_name"].astype(str)
    .str.strip()
    .str.upper()
    .str.extract(r"^([A-Z]+)", expand=False)
)

repeated_base_names = df.loc[
    df["base_name"].notna()
    & df["base_name"].duplicated(keep=False)
].sort_values(["base_name", "student_name"])

display(repeated_base_names[
    ["roll_no", "student_name", "base_name", "subject", "status"]
])

,roll_no,student_name,base_name,subject,status
19,4MW24AD020,MOHAMMED AFREED,MOHAMMED,Data Science & GenAI - 3nd Yr- AI&DS,Critical
20,4MW24AD021,MOHAMMED AWAD TONSE,MOHAMMED,Data Science & GenAI - 3nd Yr- AI&DS,Critical
18,4MW24AD019,MOHAMMED SUHAN,MOHAMMED,Data Science & GenAI - 3nd Yr- AI&DS,Critical
24,4MW24AD025,PAVAN BHAT,PAVAN,Data Science & GenAI - 3nd Yr- AI&DS,Critical
25,4MW24AD026,PAVAN S POOJARY,PAVAN,Data Science & GenAI - 3nd Yr- AI&DS,Critical
33,4MW24AD034,RAKSHITHA (ASHOK KUMAR),RAKSHITHA,Data Science & GenAI - 3nd Yr- AI&DS,Safe
34,4MW24AD035,RAKSHITHA(RAMESHA POOJARI),RAKSHITHA,Data Science & GenAI - 3nd Yr- AI&DS,Warning


In [21]:
"""Q19. Create a hypothetical Student_Details DataFrame and perform an inner join."""
student_details = pd.DataFrame({
    "roll_no": df["roll_no"],
    "branch": ["AI&DS"] * len(df),
    "semester": [5] * len(df)
})

merged_df = pd.merge(df, student_details, on="roll_no", how="inner")
display(merged_df)

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75,last_present_date,consecutive_absences,attendance_category,risk_score,classes_needed_numeric,attendance_flag,attendance_range,base_name,branch,semester
0,4MW23AD50,SRAJAN S,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11,0,0.0,Critical,33 more,NaT,11,Poor,11.0,33,Defaulter,Critical,SRAJAN,AI&DS,5
1,4MW24AD001,ADITHYA S POOJARY,Data Science & GenAI - 3nd Yr- AI&DS,11,7,4,0,64.0,Critical,5 more,2026-10-07,0,Average,1.6,5,Defaulter,Warning,ADITHYA,AI&DS,5
2,4MW24AD002,ANKITHA PRASANNA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,100.0,Safe,✓ Met,2026-10-07,0,Excellent,0.0,0,Safe,Excellent,ANKITHA,AI&DS,5
3,4MW24AD003,ANUSHA KOTEGAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91.0,Safe,✓ Met,2026-10-07,0,Excellent,0.4,0,Safe,Excellent,ANUSHA,AI&DS,5
4,4MW24AD004,ANVITA RAMESH NAIK,Data Science & GenAI - 3nd Yr- AI&DS,11,9,2,0,82.0,Warning,✓ Met,2026-10-07,0,Good,0.8,0,Safe,Good,ANVITA,AI&DS,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,4MW24AD059,ZEHA AYESHA,Data Science & GenAI - 3nd Yr- AI&DS,10,9,1,0,90.0,Safe,✓ Met,2026-10-07,0,Excellent,0.4,0,Safe,Good,ZEHA,AI&DS,5
58,4MW24AD060,AMITH G PALANDYE,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0.0,Critical,30 more,NaT,10,Poor,10.0,30,Defaulter,Critical,AMITH,AI&DS,5
59,4MW24AD061,AMOGH BHASKAR NAIK,Data Science & GenAI - 3nd Yr- AI&DS,10,5,5,0,50.0,Critical,10 more,2026-10-07,0,Poor,2.0,10,Defaulter,Critical,AMOGH,AI&DS,5
60,4MW24AD062,CHANDISH,Data Science & GenAI - 3nd Yr- AI&DS,10,6,4,0,60.0,Critical,6 more,2026-10-05,1,Average,2.2,6,Defaulter,Warning,CHANDISH,AI&DS,5


In [22]:
"""Q20. Export Critical students, status summary, and top 10 Risk Score students to CSV files."""
# (a) All Critical students
df.loc[df["status"].eq("Critical")].to_csv(
    "critical_students.csv", index=False
)

# (b) Summary statistics grouped by Status
status_export = df.groupby("status").agg(
    student_count=("roll_no", "count"),
    average_attendance_pct=("attendance_pct", "mean"),
    total_present=("present", "sum"),
    total_absent=("absent", "sum"),
    max_consecutive_absences=("consecutive_absences", "max")
)
status_export.to_csv("status_summary.csv")

# (c) Top 10 students by Risk Score
df.nlargest(10, "risk_score").to_csv(
    "top_10_risk_students.csv", index=False
)

print("Created: critical_students.csv, status_summary.csv, top_10_risk_students.csv")

Created: critical_students.csv, status_summary.csv, top_10_risk_students.csv
